In [3]:
#===========================================================================================================
#          Comprehensive BME688 Data Processing, Feature Analysis, and ML Pipeline
#===========================================================================================================
"""
Complete BME688 Sensor Data Analysis & ML Pipeline
Includes: Statistics, insight into features or Visualization, class separability, data structure for ML Models
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.signal import butter, filtfilt
from scipy.stats import skew, kurtosis

# ML & Visualization
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

# For UMAP and t-SNE
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("⚠ UMAP not installed. Install with: pip install umap-learn")

from sklearn.manifold import TSNE

# TensorFlow for Autoencoders and LSTM
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("⚠ TensorFlow not installed. Install with: pip install tensorflow")

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [7]:
#========================================
#       COMPLETE DATA COMBINING GUIDE
#========================================
"""
COMPLETE GUIDE: Combining All Separate BME688 Data Files
Shows step-by-step how to merge:
- 8 Normal measurements_act
- 6 Aceton measurements
- 6 Normal measurements_red
- 6 Redidlo measurements
- 7 Normal measurements_sav
- 7 Savo measurements
- 5 Normal measurements_soft
- 4 Softasept measurements
- 7 Normal measurements_vin
- 7 Vinegar measurements

Into one combined dataset
"""

import json
import pandas as pd
import numpy as np
import os
from pathlib import Path

print("\n" + "="*80)
print("COMPLETE DATA COMBINING GUIDE")
print("="*80)


COMPLETE DATA COMBINING GUIDE


In [8]:
# ============================================================================
# STEP 1: UNDERSTAND THE FILE STRUCTURE
# ============================================================================

print("\n[STEP 1] UNDERSTAND FILE STRUCTURE")
print("-"*80)

print("""
My current file structure should be:

my_project_folder/
├── Normal_air_1.bmerawdata
├── Normal_air_2.bmerawdata
├── Normal_air_3.bmerawdata
├── Normal_air_4.bmerawdata
├── Normal_air_5.bmerawdata
├── Normal_air_6.bmerawdata
├── Normal_air_7.bmerawdata
├── Normal_air_9.bmerawdata
├── Aceton_1.bmerawdata
├── Aceton_2.bmerawdata
├── Aceton_3.bmerawdata
├── Aceton_4.bmerawdata
├── Aceton_5.bmerawdata
├── Aceton_6.bmerawdata
├── Normal_air_11.bmerawdata
├── Normal_air_13.bmerawdata
├── Normal_air_14.bmerawdata
├── Normal_air_15.bmerawdata
├── Normal_air_16.bmerawdata
├── Normal_air_17.bmerawdata
├── Redidlo_11.bmerawdata
├── Redidlo_13.bmerawdata
├── Redidlo_14.bmerawdata
├── Redidlo_15.bmerawdata
├── Redidlo_16.bmerawdata
├── Redidlo_18.bmerawdata
├── Normal_air_21.bmerawdata
├── Normal_air_22.bmerawdata
├── Normal_air_23.bmerawdata
├── Normal_air_24.bmerawdata
├── Normal_air_25.bmerawdata
├── Normal_air_26.bmerawdata
├── Normal_air_27.bmerawdata
├── Savo_21.bmerawdata
├── Savo_22.bmerawdata
├── Savo_23.bmerawdata
├── Savo_24.bmerawdata
├── Savo_25.bmerawdata
├── Savo_26.bmerawdata
├── Savo_27.bmerawdata
├── Normal_air_31.bmerawdata
├── Normal_air_32.bmerawdata
├── Normal_air_33.bmerawdata
├── Normal_air_35.bmerawdata
├── Normal_air_36.bmerawdata
├── Softasept_31.bmerawdata
├── Softasept_34.bmerawdata
├── Softasept_35.bmerawdata
├── Softasept_36.bmerawdata
├── Normal_air_42.bmerawdata
├── Normal_air_43.bmerawdata
├── Normal_air_44.bmerawdata
├── Normal_air_45.bmerawdata
├── Normal_air_47.bmerawdata
├── Normal_air_48.bmerawdata
├── Normal_air_49.bmerawdata
├── Vinegar_42.bmerawdata
├── Vinegar_43.bmerawdata
├── Vinegar_44.bmerawdata
├── Vinegar_46.bmerawdata
├── Vinegar_47.bmerawdata
├── Vinegar_48.bmerawdata
├── Vinegar_49.bmerawdata
TOTAL: 63 files to combine
""")

# List files in current directory
print("\nFiles in current directory:")
files = os.listdir('.')
bmerawdata_files = [f for f in files if f.endswith('.bmerawdata')]
print(f"Found {len(bmerawdata_files)} .bmerawdata files")
for f in sorted(bmerawdata_files):
    print(f"  ✓ {f}")


[STEP 1] UNDERSTAND FILE STRUCTURE
--------------------------------------------------------------------------------

My current file structure should be:

my_project_folder/
├── Normal_air_1.bmerawdata
├── Normal_air_2.bmerawdata
├── Normal_air_3.bmerawdata
├── Normal_air_4.bmerawdata
├── Normal_air_5.bmerawdata
├── Normal_air_6.bmerawdata
├── Normal_air_7.bmerawdata
├── Normal_air_9.bmerawdata
├── Aceton_1.bmerawdata
├── Aceton_2.bmerawdata
├── Aceton_3.bmerawdata
├── Aceton_4.bmerawdata
├── Aceton_5.bmerawdata
├── Aceton_6.bmerawdata
├── Normal_air_11.bmerawdata
├── Normal_air_13.bmerawdata
├── Normal_air_14.bmerawdata
├── Normal_air_15.bmerawdata
├── Normal_air_16.bmerawdata
├── Normal_air_17.bmerawdata
├── Redidlo_11.bmerawdata
├── Redidlo_13.bmerawdata
├── Redidlo_14.bmerawdata
├── Redidlo_15.bmerawdata
├── Redidlo_16.bmerawdata
├── Redidlo_18.bmerawdata
├── Normal_air_21.bmerawdata
├── Normal_air_22.bmerawdata
├── Normal_air_23.bmerawdata
├── Normal_air_24.bmerawdata
├── Normal_

In [9]:
# ============================================================================
# STEP 2: CREATE HELPER FUNCTION
# ============================================================================

print("\n[STEP 2] CREATE HELPER FUNCTION TO PARSE FILES")
print("-"*80)

def parse_single_bmerawdata_file(filepath):
    """
    Parse a single .bmerawdata file and return DataFrame
    
    Args:
        filepath: Path to .bmerawdata file
    
    Returns:
        pandas.DataFrame with sensor data
    """
    
    try:
        # Load JSON
        with open(filepath, 'r') as f:
            data = json.load(f)
        
        # Extract data block and column names
        dataBlock = data['rawDataBody']['dataBlock']
        dataColumns = data['rawDataBody']['dataColumns']
        
        # Extract column names from dicts
        column_names = []
        for col in dataColumns:
            if isinstance(col, dict) and 'name' in col:
                column_names.append(col['name'])
            elif isinstance(col, str):
                column_names.append(col)
            else:
                column_names.append(str(col))
        
        # Create DataFrame
        df = pd.DataFrame(dataBlock, columns=column_names)
        
        # Convert numeric columns
        numeric_cols = ['Temperature', 'Pressure', 'Relative Humidity', 'Resistance Gassensor']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        return df
    
    except Exception as e:
        print(f"  ✗ Error parsing {filepath}: {e}")
        return None

print("✓ Helper function created")



[STEP 2] CREATE HELPER FUNCTION TO PARSE FILES
--------------------------------------------------------------------------------
✓ Helper function created


In [10]:
pip install jupyterlab

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
python.exe -m pip install --upgrade pip

SyntaxError: invalid syntax (842801469.py, line 1)